# Football-CV: NPFL Computer Vision & Analytics Pipeline
### Kaggle GPU Runner (Phase 0 Data Prep & Phase 1 YOLOv8 Fine-Tuning)

This notebook is self-contained and designed to be imported directly into **Kaggle**:
- Automatically clones/updates the `soccer_vision` codebase.
- Installs dependencies and verifies GPU acceleration.
- Downloads 1-Click Open Soccer Dataset (zero API keys required) or extracts NPFL footage.
- Remaps to 4 classes (`player`, `goalkeeper`, `referee`, `ball`) and enforces match-level train/val/test splits.
- Audits ball-to-player class distribution (<10% threshold warning).
- Calculates pitch homography matrix (camera pixels $\to$ 105m x 68m pitch coordinates).
- Fine-tunes YOLOv8 at high resolution (`imgsz=960`) to preserve small ball features.
- Validates per-class metrics (Ball AP50) and exports to ONNX.
- Copies best weights to `/kaggle/working/exported_models/` for 1-click download from Kaggle UI.

## 1. Automated Setup: Clone/Update Repo & Install Dependencies
This cell automatically detects your environment. When run on Kaggle, it clones the repository, navigates into the project directory, and verifies GPU acceleration.

In [ ]:
import os
import sys
import subprocess

# 1. Clone repository if running in fresh Kaggle environment
REPO_URL = "https://github.com/tobiebenezer/soccer_vision.git"
KAGGLE_WORKING = "/kaggle/working"

if os.path.exists(KAGGLE_WORKING):
    repo_path = os.path.join(KAGGLE_WORKING, "soccer_vision")
    if not os.path.exists(repo_path):
        print(f"[*] Cloning {REPO_URL} into {repo_path}...")
        subprocess.run(["git", "clone", REPO_URL, repo_path], check=True)
    else:
        print(f"[*] Repository already exists at {repo_path}. Pulling latest updates...")
        subprocess.run(["git", "-C", repo_path, "pull", "origin", "main"])
    os.chdir(repo_path)

print(f"[+] Current working directory: {os.getcwd()}")

# 2. Install dependencies
!pip install -q ultralytics roboflow huggingface_hub pyyaml tqdm pandas opencv-python-headless yt-dlp

# 3. Verify CUDA / GPU accelerator
import torch
print("\n================ GPU ACCELERATION CHECK ================")
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")
    print(f"Device Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("[!] WARNING: No GPU detected! Please enable GPU in Kaggle settings (Settings -> Accelerator -> GPU P100 or T4 x2)")
print("=========================================================")

## 2. API Keys & Authentication (Optional)
Option A (Open Dataset) and Option B (SoccerNet-GSR) require **NO API keys**. Only fill in if downloading custom Roboflow Universe projects.

In [ ]:
# Optional: Only required for Roboflow Universe downloads
os.environ["ROBOFLOW_API_KEY"] = ""

## 3. Phase 0 — Data Acquisition
Choose your data source:
- **Option A (Recommended & Active)**: 1-Click Open Soccer Dataset from HuggingFace (Zero API keys, ~1,000 pre-annotated frames with players, referees, and balls).
- **Option B**: SoccerNet-GSR GameState subset (`valid.zip` with 5 matches).
- **Option C**: Instant offline smoke-test data generator (60 frames in 5 seconds).
- **Option D**: NPFL match video download via YouTube (`yt-dlp`) and frame extraction.

In [ ]:
# === OPTION A (Recommended): 1-Click Open Soccer YOLO Dataset ===
# Downloads up to 1,000 pre-annotated soccer frames directly from HuggingFace (No API keys needed)
!python scripts/download_open_dataset.py --max-images 1000

# === OPTION B: SoccerNet-GSR GameState Dataset ===
# Downloads valid.zip and selectively extracts the first 5 matches
# !python scripts/download_soccernet.py --archive valid.zip --max-matches 5

# === OPTION C: Instant Offline Smoke Test (5 seconds) ===
# Generates 60 realistic soccer frames with players, goalkeepers, referees & balls
# !python scripts/create_sample_data.py --num-matches 4 --frames-per-match 15

# === OPTION D: Download & Extract NPFL Match Video ===
# Example: Download 3 minutes of an Ikorodu City or Sporting Lagos match from YouTube
# !yt-dlp -f "bestvideo[height<=1080]+bestaudio/best[height<=1080]" --download-sections "*00:15:00-00:18:00" "https://www.youtube.com/watch?v=VIDEO_ID" -o "data/raw/npfl_footage/clip.mp4"
# !python scripts/extract_frames.py --video-path data/raw/npfl_footage/clip.mp4 --venue mobolaji_johnson_arena --cam cam_main --match-id npfl_sample_01

## 4. Phase 0 — 4-Class Remapping & Manifest Generation
Consolidates raw data and maps all annotations into the team-agnostic schema:
- `0: player`
- `1: goalkeeper`
- `2: referee`
- `3: ball`

Generates `data/manifest.csv` tracking image provenance, source, and match IDs.

In [ ]:
!python scripts/merge_datasets.py

## 5. Phase 0 — Strict Match-Level Splitting
Partitions the dataset into Train (70%), Val (15%), and Test (15%) splits strictly at the **match level**.
Guarantees that all frames from any given match stay together in one split with **zero video leakage**.

In [ ]:
!python scripts/split_by_match.py --train-ratio 0.70 --val-ratio 0.15 --test-ratio 0.15

## 6. Phase 0 — Dataset Audit & Small-Object (Ball) Imbalance Check
Audits instance counts for all 4 classes. Issues a critical warning if ball instances are below 10% of player instances.

In [ ]:
!python scripts/validate_dataset.py --data-dir data/labeled

## 7. Phase 0 — Pitch Homography Calibration
Computes a 3x3 projective transformation matrix mapping camera pixels $(u, v)$ to real-world pitch meters $(X, Y)$ on a standard $105 \times 68\text{m}$ pitch.
Calibrated for Nigerian venues (e.g., Mobolaji Johnson Arena, Remo Stars Stadium).

In [ ]:
# Generates template landmark mappings and computes RANSAC homography matrix
!python scripts/calibrate_pitch.py --venue mobolaji_johnson_arena --cam cam_main

# Run with landmark point file
calib_file = "data/calibration/template_landmarks.json"
if os.path.exists(calib_file):
    !python scripts/calibrate_pitch.py --venue mobolaji_johnson_arena --cam cam_main --points-file data/calibration/template_landmarks.json

## 8. Phase 1 — Detection Model Training (YOLOv8)
Fine-tunes YOLOv8 at high resolution (`imgsz=960`) to preserve small ball pixel gradients.

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8s
model = YOLO("yolov8s.pt")

# Train model on 4-class football dataset
results = model.train(
    data="configs/dataset.yaml",
    epochs=50,
    imgsz=960,
    batch=8,
    optimizer="AdamW",
    lr0=0.001,
    mosaic=0.8,
    fliplr=0.5,
    flipud=0.0,
    name="npfl_football_cv",
    project="runs/train"
)

## 9. Phase 1 — Per-Class Metrics Inspection
Monitor class-specific AP50. Specifically watch **Class 3 (Ball)** to ensure small-object convergence.

In [ ]:
# Evaluate best checkpoint
metrics = model.val()

class_names = ["player", "goalkeeper", "referee", "ball"]
print("\n================ PER-CLASS AP50 METRICS ================")
if hasattr(metrics.box, "ap50"):
    for idx, name in enumerate(class_names):
        if idx < len(metrics.box.ap50):
            print(f"  [{idx}] {name:<12}: AP50 = {metrics.box.ap50[idx]:.4f}")
print(f"  Aggregate mAP50    : {metrics.box.map50:.4f}")
print(f"  Aggregate mAP50-95 : {metrics.box.map:.4f}")
print("=========================================================")

## 10. Phase 1 — Export to ONNX & Package Weights for Kaggle Download
Exports model to ONNX format and copies both `.pt` and `.onnx` weights into `/kaggle/working/exported_models/` for immediate 1-click download from the Kaggle UI output panel.

In [ ]:
import shutil

# 1. Export fine-tuned weights to ONNX format
onnx_path = model.export(format="onnx", imgsz=960, dynamic=True)
print(f"[+] Exported ONNX model to: {onnx_path}")

# 2. Copy models to Kaggle working output directory for 1-click download
export_dir = "/kaggle/working/exported_models" if os.path.exists("/kaggle/working") else "exported_models"
os.makedirs(export_dir, exist_ok=True)

best_pt = "runs/train/npfl_football_cv/weights/best.pt"
best_onnx = "runs/train/npfl_football_cv/weights/best.onnx"

if os.path.exists(best_pt):
    shutil.copy2(best_pt, os.path.join(export_dir, "best.pt"))
    print(f"[+] Copied best.pt -> {export_dir}/best.pt")

if os.path.exists(best_onnx):
    shutil.copy2(best_onnx, os.path.join(export_dir, "best.onnx"))
    print(f"[+] Copied best.onnx -> {export_dir}/best.onnx")

print("\n=========================================================")
print(f"[SUCCESS] Pipeline Completed!")
print(f"Trained models are ready for download in Kaggle Output tab: {export_dir}")
print("=========================================================")